In [20]:
import pg8000
from dotenv import load_dotenv
import os
from sqlalchemy import create_engine, MetaData, Table, select, insert
from sqlalchemy.exc import SQLAlchemyError
load_dotenv()
import pandas as pd

In [21]:
DB_HOST = os.getenv("DB_HOST")
NC_ACC = os.getenv("NC_ACC")
NC_PASS = os.getenv("NC_PASS")

def create_db_connection():
    DB_HOST = os.getenv("DB_HOST")
    DB_NAME = os.getenv("DB_NAME")
    DB_USER = os.getenv("DB_USER")
    DB_PASSWORD = os.getenv("DB_PASSWORD")
    engine = create_engine('postgresql+pg8000://'+DB_USER+':'+DB_PASSWORD+'@'+DB_HOST+':5432/'+DB_NAME)
    return engine

In [22]:
engine = create_db_connection()
id= 2158
       
df_hypo = pd.read_sql(f'''SELECT *
                            FROM bre_advance_search
                            WHERE bre_advance_search.id = '{id}' ''', engine)
df_tags = pd.read_sql(f'''SELECT oc_systemtag.name AS tag_name
                                    FROM oc_systemtag_object_mapping
                                    JOIN oc_systemtag ON oc_systemtag_object_mapping.systemtagid = oc_systemtag.id
                                    WHERE oc_systemtag_object_mapping.objectid = '{id}' ''', engine)

In [23]:
df_tags

,tag_name
0,flower
1,nature
2,still
3,life
4,vase
5,red
6,white
7,orange
8,green
9,bloom


In [24]:
df_hypo

,index,id,is_color,is_adjective,lvl3_hyponym,hyponym_all


In [ ]:
    for idx, col in enumerate(cols):
        with col:
            len_col = len(df_data) // N_of_cols
            start_idx = idx * len_col
            end_idx = (idx + 1) * len_col if idx != N_of_cols - 1 else len(df_data)
            display_data = df_data[start_idx:end_idx]
            for index, row in display_data.iterrows():
                try:
                    file_id = row['fileid']
                    img_path = row['preview_url']
                    tag_names = row.get('tagnames', '')

                    st.image(get_images(file_id, img_path))
                    st.write(f"{tag_names}")
                except Exception as e:
                    print(f"Error loading image for file_id {file_id}: {e}")
       #try:

           # st.write(f"hypo: ")
           # st.dataframe(get_hypo_search(file_id))
        #except Exception as e:
         #   print(f"no Preview availible for:{img_path} /n: {e}")


In [ ]:
def get_images(file_id,file_path):
    DB_HOST = os.getenv("DB_HOST")
    NC_ACC = os.getenv("NC_ACC")
    NC_PASS = os.getenv("NC_PASS")

    server_url = f'''http://{DB_HOST}:8080/remote.php/dav/files/{NC_ACC}'''
    preview_url = f'''http://{DB_HOST}:8080/core/preview?fileId={file_id}&x=1080&y=1080'''
    username = NC_ACC
    password = NC_PASS

    # Send a GET request to download the file
    response = requests.get(preview_url, auth=HTTPBasicAuth(username, password), stream=True)
 
    # Check if the request was successful
    if response.status_code == 200:
        # Save the file content in memory using BytesIO
        file_in_memory = BytesIO()
        for chunk in response.iter_content(chunk_size=1024):
            if chunk:
                file_in_memory.write(chunk)
        # Open the image using Pillow (PIL)
        img = PILImage.open(file_in_memory)
        return img ,file_id
    elif response.status_code == 404:
        print(f"no preview availible for file: {file_id} {file_path}")
    else:
        print(f"Failed to download file. Status code: {response.status_code}")
        print(response.text)
    try:
        import gc
        del file_in_memory
        gc.collect()
    except:
        pass